# **Week 3: Sequence Models and Attention Mechanism**

Augment your sequence models using an attention mechanism, an algorithm that helps your model decide where to focus its attention given a sequence of inputs. Then, explore speech recognition and how to deal with audio data.

**Learning Objectives:**

- Describe a basic sequence-to-sequence model
- Compare and contrast several different algorithms for language translation
- Optimize beam search and analyze it for errors
- Use beam search to identify likely translations
- Apply BLEU score to machine-translated text
- Implement an attention model
- Train a trigger word detection model and make predictions
- Synthesize and process audio recordings to create train/dev datasets
- Structure a speech recognition project

---

## **Table of Contents**

---

## **Basic Models**

This section provides an introduction to **Sequence-to-Sequence (Seq2Seq)** architectures, which have revolutionized how we handle data that varies in length, such as sentences or audio clips. The core takeaway is the "Encoder-Decoder" framework: one part of the network understands the input, and the other part expresses the output.


### **High-level Summary**

We are looking at a fundamental shift in how neural networks handle data. In simpler models, we often had a fixed input size and a fixed output size. However, in tasks like **Machine Translation**, the input sentence "Jane visite l'Afrique en Septembre" (5 words in French) might result in an English output of 6 words. 

To bridge this gap, we use an **Encoder** to act as a "translator's brain," absorbing the full context of the input and condensing it into a dense mathematical vector. This vector is the "thought" that is then passed to the **Decoder**, which has the job of "speaking" that thought in the target language.

Interestingly, this logic extends beyond text. By swapping the RNN encoder for a **CNN**, we can effectively "translate" an image into a caption. The CNN sees a cat on a chair and turns those pixels into a feature vector; the RNN then takes that vector and generates the phrase "a cat sitting on a chair." The overarching magic here is that as long as we can turn an input into a representative vector, we can train an RNN to turn that vector into a sequence of meaningful words.


### **Key Bullet Points**

* **The Encoder-Decoder Framework:**
    * **Encoder:** An RNN (often using GRU or LSTM units) processes the input sequence (e.g., a French sentence) and compresses it into a single representational vector.
    * **Decoder:** A separate RNN takes that vector as its starting point and generates the output sequence (e.g., the English translation) one word at a time until it hits an "End of Sequence" token.

<div align='center'>
<img src='assets/seq2seq-mt.png' width=500px>
</div>

* **Versatility of Inputs:** 
    * The "Encoder" doesn't have to be an RNN. For image captioning, we use a CNN (like a pre-trained AlexNet) to extract a feature vector from an image.
    * This feature vector is then fed into an RNN decoder to "describe" the image in natural language.

<div align='center'>
<img src='assets/seq2seq-ic.png' width=750px>
</div>

* **The Goal of "Best" vs. "Random":** * Unlike basic language models that might randomly sample words to generate creative text, Seq2Seq tasks like translation require finding the **most likely** sequence (the best translation), not just any sequence.

---

## **Picking the Most Likely Sentence**

This section builds upon the basic Seq2Seq architecture by framing it as a **Conditional Language Model** and explaining the search algorithms required to make it functional. The primary focus shifts from simply building the network to the computational challenge of finding the most accurate output.

### **High-level Summary**

We can see a fine line between the "generative" language models we often use for creative text and the "conditional" models needed for translation. In essence, the decoder is just a language model that has been "primed" with a specific thought from the encoder. Instead of asking the model, "What is a likely sentence to exist?", we are asking, "What is the most likely English version of *this* specific French idea?"

A significant portion of the discussion focuses on why we can't just take the easy way out with **Greedy Search**. Because language is a sequence where every choice impacts the future, picking the best word at the moment is like making a short-sighted chess move—it might look good now but ruin the "endgame" of the sentence. However, the search space for a perfect sentence is too massive for even the most powerful computers to check every possibility. This sets the stage for **Beam Search**, which acts as a middle ground: it’s more sophisticated than greedy search but far more efficient than checking every possible sentence in the dictionary.


### **Key Bullet Points**

* **Machine Translation as a "Conditional" Model:** 
    * A standard language model estimates the probability of any given sentence: $P(y^{<1>}, \dots, y^{<T_y>})$.
    * A machine translation model estimates the probability of an English sentence **conditioned** on a specific French sentence: $P(y^{<1>}, \dots, y^{<T_y>} \mid x)$ and the goal is to find a $y$ to maximize this conditional probablity.
    * In a basic language model, the sequence starts with a vector of zeros. 
    * In Seq2Seq, the decoder starts with a **context vector** provided by the encoder, which represents the meaning of the input sentence.

<div align='center'>
<img src='assets/conditional_lm.png' width=750px>
</div>

* **Maximization vs. Random Sampling:**
    * Unlike generative language models (where we might sample words randomly for creativity), translation requires finding the single most likely sentence that maximizes the conditional probability.
* **The Failure of "Greedy Search":**
    * Greedy search picks the best word at step 1, then the best at step 2, and so on. 
    * This often fails because a "good" first word (e.g., "going") might lead the model down a path toward a suboptimal, verbose sentence, whereas a slightly "less likely" early word (e.g., "visiting") could lead to a better overall translation.
    * The total number of possible English sentences is astronomically large (e.g., $10,000^{10}$ for a 10-word sentence). 
    * Since we cannot check every combination, we must use an approximate search algorithm like **Beam Search** to find a "good enough" maximum.